# Step 3: BPR 기반 Projection/Fusion Tuning

이 노트북은 frozen 상태의 사전 계산된 text/image/tabular feature bank를 입력으로 사용하고, BPR loss를 통해 user embedding, text projection, image projection, tabular projection, fusion MLP를 학습합니다. 이후 Step 1, Step 2 산출물과 같은 형식의 64D game embedding bank를 저장합니다.

입력:
- Step 1 산출물: `game_fusion/emb_game_concat_64.npy`
- Step 2 산출물: `game_fusion/emb_game_text_only_64.npy`, `game_fusion/emb_game_image_only_64.npy`, `game_fusion/emb_game_tabular_only_64.npy`
- `text_data/emb_text_minilm` feature bank
- `image_embedding/emb_clip_squash` feature bank
- `tabular_embedding/emb_tabular_svd64` feature bank

출력:
- `game_fusion/emb_game_finetuned_64.npy`
- `game_fusion/emb_game_finetuned_64.csv`


In [1]:
import importlib
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
if ROOT.name == "game_fusion":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
NUM_EPOCHS = 100
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Root path: {ROOT}")
print(f"PyTorch device: {DEVICE}")


Root path: c:\Users\User\26_2_Contest
PyTorch device: cpu


## 1. Step 1/2 산출물과 Feature Bank 로드

Step 1/2 embedding을 로드해 흐름이 이어지는지 확인하고 비교 기준으로 사용합니다. 학습 입력으로는 frozen raw feature bank를 사용하며, game catalog의 `app_id` 순서에 맞춰 text/image/tabular bank를 정렬합니다.


In [3]:
from tabular_embedding.tabular_tower import load_tabular_bank
import game_fusion.fusion_tower as fusion_tower_module

importlib.reload(fusion_tower_module)
from game_fusion.fusion_tower import BPRModel, prepare_bpr_data, train_epoch_bpr

text_tower_path = ROOT / "text_data" / "08_text_tower.py"
text_spec = importlib.util.spec_from_file_location("text_tower", text_tower_path)
text_tower_module = importlib.util.module_from_spec(text_spec)
text_spec.loader.exec_module(text_tower_module)
load_text_bank = text_tower_module.load_text_bank

image_tower_path = ROOT / "image_embedding" / "07_image_tower.py"
image_spec = importlib.util.spec_from_file_location("image_tower", image_tower_path)
image_tower_module = importlib.util.module_from_spec(image_spec)
image_spec.loader.exec_module(image_tower_module)
load_image_bank = image_tower_module.load_image_bank

games = pd.read_parquet(ROOT / "Data_process" / "games_metadata_enriched.parquet")
game_ids = games["app_id"].to_numpy()

emb_concat_step1 = np.load(ROOT / "game_fusion" / "emb_game_concat_64.npy").astype(np.float32)
emb_text_only = np.load(ROOT / "game_fusion" / "emb_game_text_only_64.npy").astype(np.float32)
emb_image_only = np.load(ROOT / "game_fusion" / "emb_game_image_only_64.npy").astype(np.float32)
emb_tab_only = np.load(ROOT / "game_fusion" / "emb_game_tabular_only_64.npy").astype(np.float32)

text_bank, text_id2row = load_text_bank(
    ROOT / "text_data" / "emb_text_minilm",
    app_ids=game_ids,
    device=DEVICE,
    fill_missing=True,
)
image_bank, image_id2row = load_image_bank(
    ROOT / "image_embedding" / "emb_clip_squash",
    app_ids=game_ids,
    device=DEVICE,
    fill_missing=True,
)
tab_bank, tab_id2row = load_tabular_bank(
    ROOT / "tabular_embedding" / "emb_tabular_svd64",
    app_ids=game_ids,
    device=DEVICE,
    fill_missing=True,
)

print("Loaded game catalog and Step 1/2 outputs")
print(f"  Games: {len(game_ids):,}")
print(f"  Step 1 text+image+tabular concat: {emb_concat_step1.shape}")
print(f"  Step 2 text-only: {emb_text_only.shape}")
print(f"  Step 2 image-only: {emb_image_only.shape}")
print(f"  Step 2 tabular-only: {emb_tab_only.shape}")
print(f"  Text bank: {tuple(text_bank.shape)}")
print(f"  Image bank: {tuple(image_bank.shape)}")
print(f"  Tabular bank: {tuple(tab_bank.shape)}")


[image_tower] Missing images filled with mean vector: 8 items, sample=[np.int64(2381590), np.int64(661700), np.int64(451330), np.int64(1204870), np.int64(1398280)]
Loaded game catalog and Step 1/2 outputs
  Games: 50,872
  Step 1 text+image+tabular concat: (50872, 64)
  Step 2 text-only: (50872, 64)
  Step 2 image-only: (50872, 64)
  Step 2 tabular-only: (50872, 64)
  Text bank: (50872, 384)
  Image bank: (50872, 512)
  Tabular bank: (50872, 64)


## 2. Interaction Data 준비

recommendation/interactions CSV가 없으면 작은 synthetic dataset을 만들어 pipeline을 end-to-end로 smoke test할 수 있게 합니다.

In [4]:
interaction_files = sorted((ROOT / "mvp_recommendation").glob("*recommendations*.csv"))

if interaction_files:
    interactions_df = pd.read_csv(interaction_files[0])
    print(f"Loaded interactions from: {interaction_files[0]}")
else:
    print("Warning: no interaction CSV found. Creating synthetic data for smoke testing.")
    n_users = 100
    n_interactions = 500
    game_indices = np.random.choice(len(game_ids), n_interactions)
    interactions_df = pd.DataFrame({
        "user_id": np.random.choice(n_users, n_interactions),
        "app_id": game_ids[game_indices],
        "is_recommended": np.random.rand(n_interactions) > 0.3,
        "date": pd.date_range("2024-01-01", periods=n_interactions, freq="h"),
        "hours": np.random.exponential(10, n_interactions),
        "review_id": np.arange(n_interactions),
    })

if "is_recommended" in interactions_df.columns:
    positive_interactions = interactions_df[interactions_df["is_recommended"] == True].copy()
else:
    positive_interactions = interactions_df.copy()

positive_interactions = positive_interactions[positive_interactions["app_id"].isin(game_ids)].copy()

user_ids_unique = sorted(interactions_df["user_id"].unique())
user_id2idx = {uid: idx for idx, uid in enumerate(user_ids_unique)}
game_id2idx = {gid: idx for idx, gid in enumerate(game_ids)}

print("Interaction summary")
print(f"  Total rows: {len(interactions_df):,}")
print(f"  Positive rows: {len(positive_interactions):,}")
print(f"  Users: {len(user_id2idx):,}")
print(f"  Games in catalog: {len(game_id2idx):,}")


Interaction summary
  Total rows: 500
  Positive rows: 372
  Users: 100
  Games in catalog: 50,872


## 3. BPR Sample 생성

각 sample은 `(user_idx, positive_game_idx, negative_game_idx)` 형태입니다. 가능한 경우 negative game은 같은 user가 positive interaction을 남기지 않은 game에서 샘플링합니다.

In [5]:
bpr_data = prepare_bpr_data(
    positive_interactions,
    len(game_ids),
    game_id2idx,
    user_id2idx,
)

if not bpr_data:
    print("Warning: no usable positives. Creating dummy BPR samples.")
    num_dummy_users = max(len(user_id2idx), 1)
    bpr_data = [
        (np.random.randint(num_dummy_users), np.random.randint(len(game_ids)), np.random.randint(len(game_ids)))
        for _ in range(100)
    ]

user_indices = torch.tensor([row[0] for row in bpr_data], dtype=torch.long)
pos_game_indices = torch.tensor([row[1] for row in bpr_data], dtype=torch.long)
neg_game_indices = torch.tensor([row[2] for row in bpr_data], dtype=torch.long)

train_dataset = TensorDataset(user_indices, pos_game_indices, neg_game_indices)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print(f"BPR samples: {len(bpr_data):,}")
print(f"Batches per epoch: {len(train_loader):,}")


BPR samples: 372
Batches per epoch: 2


## 4. BPR Model 초기화 및 학습

`BPRModel`은 user embedding과 text/image/tabular/fusion projection tower를 함께 학습합니다. MiniLM/CLIP/SVD feature bank 자체는 frozen input으로 유지하고, 각 projection tower와 fusion tower만 BPR objective에 맞춰 조정합니다.


In [6]:
num_users = max(len(user_id2idx), 1)
model = BPRModel(num_users=num_users, embed_dim=64).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("BPR model initialized")
print(f"  Users: {num_users:,}")
print("  Modalities: text + image + tabular")
print(f"  Trainable params: {trainable_params:,}")

train_losses = []
for epoch in range(NUM_EPOCHS):
    loss = train_epoch_bpr(model, train_loader, optimizer, text_bank, image_bank, tab_bank, DEVICE)
    train_losses.append(loss)
    scheduler.step()

    if epoch == 0 or (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:03d}/{NUM_EPOCHS} | loss={loss:.6f}")

print(f"Training complete. Final loss: {train_losses[-1]:.6f}")


BPR model initialized
  Users: 100
  Modalities: text + image + tabular
  Trainable params: 356,800
Epoch 001/100 | loss=0.694608
Epoch 010/100 | loss=0.389243
Epoch 020/100 | loss=0.271072
Epoch 030/100 | loss=0.232487
Epoch 040/100 | loss=0.217335
Epoch 050/100 | loss=0.211516
Epoch 060/100 | loss=0.206830
Epoch 070/100 | loss=0.206866
Epoch 080/100 | loss=0.205371
Epoch 090/100 | loss=0.204572
Epoch 100/100 | loss=0.205631
Training complete. Final loss: 0.205631


## 5. Fine-Tuned Game Embedding 생성 및 저장

학습이 끝난 뒤 export할 game bank에는 user embedding이 필요하지 않습니다. `forward_game_only`로 모든 game에 대해 학습된 text/image/tabular/fusion 경로만 실행합니다.


In [7]:
model.eval()
all_embeddings = []

with torch.no_grad():
    for start_idx in range(0, len(game_ids), BATCH_SIZE):
        end_idx = min(start_idx + BATCH_SIZE, len(game_ids))
        z_text = text_bank[start_idx:end_idx].to(DEVICE)
        z_image = image_bank[start_idx:end_idx].to(DEVICE)
        z_tab = tab_bank[start_idx:end_idx].to(DEVICE)
        game_emb = model.forward_game_only(z_text, z_image, z_tab)
        all_embeddings.append(game_emb.cpu().numpy())

game_embeddings_finetuned = np.concatenate(all_embeddings, axis=0).astype(np.float32)

output_dir = ROOT / "game_fusion"
emb_path = output_dir / "emb_game_finetuned_64.npy"
csv_path = output_dir / "emb_game_finetuned_64.csv"

np.save(emb_path, game_embeddings_finetuned)
pd.DataFrame({"app_id": game_ids}).to_csv(csv_path, index=False)

norms = np.linalg.norm(game_embeddings_finetuned, axis=1)
print("Saved Step 3 embeddings")
print(f"  NPY: {emb_path}")
print(f"  CSV: {csv_path}")
print(f"  Shape: {game_embeddings_finetuned.shape}")
print(f"  Dtype: {game_embeddings_finetuned.dtype}")
print(f"  Norm mean/std: {norms.mean():.6f} / {norms.std():.6f}")


Saved Step 3 embeddings
  NPY: c:\Users\User\26_2_Contest\game_fusion\emb_game_finetuned_64.npy
  CSV: c:\Users\User\26_2_Contest\game_fusion\emb_game_finetuned_64.csv
  Shape: (50872, 64)
  Dtype: float32
  Norm mean/std: 1.000000 / 0.000000


## 6. Step 1과 Step 3 Embedding Geometry 비교

이 비교는 recommendation metric이 아니라 가벼운 sanity check입니다. Step 1은 frozen text/image/tabular concat fusion이고, Step 3은 같은 입력 모달리티를 BPR objective로 조정한 결과입니다. 실제 Recall/NDCG는 downstream recommendation pipeline에서 측정해야 합니다.


In [8]:
sample_size = min(100, len(game_ids))
sample_games = np.random.choice(len(game_ids), sample_size, replace=False)

sim_step1 = cosine_similarity(emb_concat_step1[sample_games]).flatten()
sim_step3 = cosine_similarity(game_embeddings_finetuned[sample_games]).flatten()
sim_step1 = sim_step1[sim_step1 != 1.0]
sim_step3 = sim_step3[sim_step3 != 1.0]

print("Embedding geometry comparison")
print(f"  Step 1 frozen text+image+tabular concat: mean={sim_step1.mean():.6f}, std={sim_step1.std():.6f}")
print(f"  Step 3 BPR tuned:                     mean={sim_step3.mean():.6f}, std={sim_step3.std():.6f}")
print("Step 3 complete")


Embedding geometry comparison
  Step 1 frozen text+image+tabular concat: mean=0.855415, std=0.050752
  Step 3 BPR tuned:                     mean=0.021721, std=0.292186
Step 3 complete
